In [1]:
import sys

print(sys.executable)

C:\Users\USUARIO\AppData\Local\pypoetry\Cache\virtualenvs\soja-yield-prediction-UvT_ZK9U-py3.13\Scripts\python.exe


In [2]:
import pandas as pd

df = pd.read_excel("../data/raw/magyp/estimaciones_agricolas_magyp.xlsx")
df.shape

(160499, 11)

In [3]:
df.columns.tolist()

['cultivo',
 'anio',
 'campania',
 'provincia',
 'provincia_id',
 'departamento',
 'departamento_id',
 'superficie_sembrada_ha',
 'superficie_cosechada_ha',
 'produccion_tm',
 'rendimiento_kgxha']

In [4]:
df.dtypes

cultivo                        str
anio                         int64
campania                       str
provincia                      str
provincia_id               float64
departamento                   str
departamento_id            float64
superficie_sembrada_ha       int64
superficie_cosechada_ha      int64
produccion_tm                int64
rendimiento_kgxha            int64
dtype: object

In [5]:
df.head()

,cultivo,anio,campania,provincia,provincia_id,departamento,departamento_id,superficie_sembrada_ha,superficie_cosechada_ha,produccion_tm,rendimiento_kgxha
0,ajo,1969,1969/1970,Buenos Aires,6.0,25 de Mayo,6854.0,3,3,10,3333
1,ajo,1969,1969/1970,Buenos Aires,6.0,Adolfo Gonzales Chaves,6014.0,15,15,82,5467
2,ajo,1969,1969/1970,Buenos Aires,6.0,Almirante Brown,6028.0,2,2,8,4000
3,ajo,1969,1969/1970,Buenos Aires,6.0,Balcarce,6063.0,450,450,2025,4500
4,ajo,1969,1969/1970,Buenos Aires,6.0,Cañuelas,6134.0,2,2,7,3500


In [6]:
df["cultivo"].unique()

<StringArray>
[             'ajo',          'algodón',          'alpiste',
            'arroz',           'arveja',            'avena',
           'banana',   'caña de azúcar',          'cártamo',
 'cebada cervecera', 'cebada forrajera',     'cebada total',
    'cebolla total',          'centeno',            'colza',
         'garbanzo',          'girasol',           'jojoba',
          'lenteja',            'limón',             'lino',
             'maíz',        'mandarina',             'maní',
             'mijo',          'naranja',       'papa total',
           'pomelo',    'poroto alubia',     'poroto negro',
     'poroto otros',     'poroto total',         'soja 1ra',
         'soja 2da',       'soja total',            'sorgo',
               'té',    'trigo candeal',      'trigo total',
             'tung',       'yerba mate']
Length: 41, dtype: str

In [7]:
df_soja = df[df["cultivo"].isin(["soja 1ra", "soja 2da"])]
df_soja.shape

(11585, 11)

In [8]:
df_soja["anio"].min(), df_soja["anio"].max()

(np.int64(2000), np.int64(2024))

In [9]:
df_soja.isnull().sum()

cultivo                    0
anio                       0
campania                   0
provincia                  0
provincia_id               0
departamento               0
departamento_id            0
superficie_sembrada_ha     0
superficie_cosechada_ha    0
produccion_tm              0
rendimiento_kgxha          0
dtype: int64

In [10]:
df_provincial = df_soja.groupby(
    ["provincia", "campania", "cultivo"], as_index=False
).agg(
    superficie_sembrada_ha=("superficie_sembrada_ha", "sum"),
    superficie_cosechada_ha=("superficie_cosechada_ha", "sum"),
    produccion_tm=("produccion_tm", "sum"),
)

df_provincial["rendimiento_kgxha"] = (
    df_provincial["produccion_tm"] * 1000 / df_provincial["superficie_cosechada_ha"]
)

df_provincial.shape

(672, 7)

In [11]:
df_provincial = df_soja.groupby(
    ["provincia", "campania", "cultivo"], as_index=False
).agg(
    superficie_sembrada_ha=("superficie_sembrada_ha", "sum"),
    superficie_cosechada_ha=("superficie_cosechada_ha", "sum"),
    produccion_tm=("produccion_tm", "sum"),
)

df_provincial["rendimiento_kgxha"] = (
    df_provincial["produccion_tm"] * 1000 / df_provincial["superficie_cosechada_ha"]
)

df_provincial.shape

(672, 7)

In [14]:
df_provincial["provincia"].unique()

<StringArray>
[       'Buenos Aires',           'Catamarca',               'Chaco',
          'Corrientes',             'Córdoba',          'Entre Ríos',
             'Formosa',               'Jujuy',            'La Pampa',
            'Misiones',               'Salta',            'San Luis',
            'Santa Fe', 'Santiago del Estero',             'Tucumán']
Length: 15, dtype: str

In [16]:
df_provincial.groupby("provincia")["campania"].nunique()

provincia
Buenos Aires           25
Catamarca              25
Chaco                  25
Corrientes             25
Córdoba                25
Entre Ríos             25
Formosa                25
Jujuy                  24
La Pampa               25
Misiones               25
Salta                  25
San Luis               25
Santa Fe               25
Santiago del Estero    25
Tucumán                25
Name: campania, dtype: int64

In [1]:
from soja_yield_prediction.data.carga_magyp import cargar_soja_provincial

df_provincial = cargar_soja_provincial(
    "../data/raw/magyp/estimaciones_agricolas_magyp.xlsx"
)

df_provincial.shape

(672, 7)

In [2]:
df_provincial.to_csv("../data/interim/soja_rendimiento_provincial.csv", index=False)